# Random Forest ile RUL Tahmini

## 1. Veri Yükleme ve İnceleme

Bu çalışmada NASA N-CMAPSS DS02 veri seti kullanılmaktadır.

Amaç, motorun çalışma koşulları ve fiziksel sensör
ölçümlerinden Remaining Useful Life (RUL) tahmin etmektir.

Ortak preprocessing:
- Window size: 256
- Stride: 1024
- Input features: 18
- Validation: Leave-One-Engine-Out

In [11]:
from pathlib import Path 
import sys 

import numpy as np 
import pandas as pd 

current_dir = Path.cwd().resolve()

# proje ana klasörünü bulalım 
if (current_dir / "preprocessing").is_dir():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "preprocessing").is_dir():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "Proje ana klasörü bulunamadı."
    )

# Ortak preprocessing modüllerine erişim
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)



Project root: /Users/helinkesip/Desktop/n_cmapss_guncel/n_cmapss


In [12]:
from preprocessing.data_loader import (
    NCmapssDataLoader,
    load_window_metadata,
    load_feature_names
)

# Preprocessing tarafından oluşturulan pencere bilgilerini okuyalım
windows = load_window_metadata()

print("Toplam pencere sayısı:", len(windows))

# İlk pencere
first_window = windows[0]

print("\nİlk pencerenin bilgileri:")
print(first_window)

# İlk pencerenin gerçek sensör ölçümlerini yükleyelim
with NCmapssDataLoader() as loader:
    X_window, y_rul = loader.load_window(first_window)

# Sensör isimleri
feature_names = load_feature_names()

# sensör framelerini dataframe formatına çevirelim 
df_window = pd.DataFrame(
    X_window,
    columns=feature_names
)

print("\nPencere boyutu:", X_window.shape)
print("RUL değeri:", y_rul)

display(df_window.head(10))

Toplam pencere sayısı: 5270

İlk pencerenin bilgileri:
WindowRecord(window_id=0, unit=2, cycle=1, rul=74.0, start_row=0, end_row=256)

Pencere boyutu: (256, 18)
RUL değeri: 74.0


,alt,Mach,TRA,T2,T24,T30,T48,T50,P15,P2,P21,P24,Ps30,P40,P50,Nf,Nc,Wf
0,10005.0,0.448497,76.903748,502.420929,600.148010,1438.498169,1818.027710,1228.129883,15.806267,11.577097,16.046970,20.126623,331.293671,336.631836,12.629360,2160.926514,8591.373047,3.855337
1,10013.0,0.447741,76.903748,502.326111,600.055908,1438.350220,1817.682617,1227.879150,15.795477,11.568235,16.036016,20.113218,331.109863,336.446747,12.623033,2160.909424,8590.972656,3.852319
2,10017.0,0.448938,77.079529,502.416077,600.210754,1439.109131,1820.020630,1229.422485,15.807747,11.574866,16.048473,20.130957,331.753174,337.082489,12.637951,2161.819092,8593.032227,3.866201
3,10024.0,0.449883,77.079529,502.469879,600.369690,1439.240234,1819.188354,1228.538696,15.816360,11.578198,16.057219,20.146717,331.819122,337.162842,12.631510,2162.768555,8593.781250,3.863328
4,10031.0,0.449379,77.079529,502.401276,600.298218,1439.063965,1818.963501,1228.389038,15.807512,11.571593,16.048237,20.135889,331.626007,336.966949,12.624871,2162.619629,8593.219727,3.860818
5,10040.0,0.447993,77.079529,502.248779,600.132263,1438.787598,1818.564697,1228.168945,15.790096,11.557944,16.030554,20.113279,331.319000,336.655396,12.616795,2162.391357,8592.379883,3.856593
6,10044.0,0.448749,77.079529,502.297394,600.153015,1438.802368,1818.881104,1228.441284,15.792666,11.561284,16.033165,20.115416,331.339630,336.672638,12.618247,2162.172607,8592.301758,3.858025
7,10048.0,0.449820,77.079529,502.376495,600.241394,1438.947021,1818.997314,1228.446289,15.800091,11.567045,16.040701,20.125383,331.464935,336.800629,12.619357,2162.366943,8592.770508,3.859438
8,10056.0,0.448938,77.079529,502.269928,600.148315,1438.776245,1818.528442,1228.084351,15.788539,11.557252,16.028975,20.111582,331.255585,336.591125,12.611525,2162.413330,8592.335938,3.855745
9,10059.0,0.448875,77.079529,502.253998,600.111267,1438.689209,1818.607178,1228.197266,15.785221,11.555538,16.025604,20.106560,331.174164,336.506134,12.610264,2162.179688,8591.984375,3.855408


### Özellik Vektörünün Oluşturulması (Feature Vector)

N-CMAPSS DS02 veri setinden elde edilen her pencere, belirli bir zaman aralığındaki sensör ölçümlerini içermektedir.

Random Forest Regressor, her örneğin sabit uzunlukta bir özellik vektörüyle temsil edilmesini gerektirir.
Her sensör için aşağıdaki dört özellik hesaplanmaktadır:

- **Mean:** Pencere boyunca sensör ölçümlerinin ortalaması.
- **Standard Deviation:** Ölçümlerin ortalama etrafındaki değişkenliği.
- **Minimum:** Pencere içerisindeki en düşük ölçüm.
- **Maximum:** Pencere içerisindeki en yüksek ölçüm.

Bu işlem sonucunda her pencere, sensör sayısının dört katı uzunluğunda bir özellik vektörüne dönüştürülmektedir.
Bu özellikler 

Böylece zaman serisi verileri, Random Forest algoritmasının işleyebileceği tablosal veri formatına dönüştürülmüş olur.

In [13]:
# İlk pencerenin sensör istatistikleri

window_features = df_window.agg(  #birden fazla istatistik hesaplamak için agg kullanıyoruz
    ["mean", "std", "min", "max"]
)

display(window_features)

,alt,Mach,TRA,T2,T24,T30,T48,T50,P15,P2,P21,P24,Ps30,P40,P50,Nf,Nc,Wf
mean,10619.550781,0.477126,77.315735,502.723663,600.773621,1438.587402,1817.373291,1224.741211,15.713846,11.509169,15.953144,20.041819,328.911865,334.223328,12.408353,2165.346191,8591.742188,3.824599
std,334.872070,0.020051,0.211382,0.643623,0.879924,0.973423,0.986034,1.796483,0.034115,0.024257,0.034635,0.045162,0.987048,0.995116,0.111826,3.233718,3.135091,0.013563
min,10005.000000,0.447741,76.903748,501.555542,599.262695,1436.715576,1815.256714,1221.143921,15.636933,11.455660,15.875059,19.948034,326.673798,331.965546,12.170094,2160.716797,8585.840820,3.795418
max,11309.000000,0.511749,77.782646,503.632019,602.041687,1440.118286,1820.020630,1229.422485,15.816360,11.578198,16.057219,20.146717,331.819122,337.162842,12.637951,2171.427490,8596.522461,3.866201


In [14]:
# İstatistiksel özellikleri tek satırlık vektöre dönüştürelim
feature_vector = window_features.T

# Sensör ve istatistik isimlerini sütunlara taşıyalım
feature_vector = feature_vector.stack().to_frame().T

# Sütun isimlerini oluşturalım
feature_vector.columns = [
    f"{sensor}_{stat}"
    for sensor, stat in feature_vector.columns
]

print("Özellik vektörünün boyutu:", feature_vector.shape)

display(feature_vector.iloc[:, :12])

Özellik vektörünün boyutu: (1, 72)


,alt_mean,alt_std,alt_min,alt_max,Mach_mean,Mach_std,Mach_min,Mach_max,TRA_mean,TRA_std,TRA_min,TRA_max
0,10619.550781,334.87207,10005.0,11309.0,0.477126,0.020051,0.447741,0.511749,77.315735,0.211382,76.903748,77.782646


### 3. Tüm Pencereler İçin Özellik Çıkarımı

Önceki aşamada tek bir pencerenin sensör ve çalışma koşulu değişkenlerinden 72 istatistiksel özellik elde edilmiştir.

Bu aşamada aynı işlem, preprocessing tarafından oluşturulan tüm pencerelere uygulanacaktır.

Her pencere için:

- Sensör ölçümleri yüklenecektir.
- Ortalama, standart sapma, minimum ve maksimum değerleri hesaplanacaktır.
- Hesaplanan özellikler tek satırlık bir özellik vektörüne dönüştürülecektir.
- Pencereye karşılık gelen gerçek RUL değeri kaydedilecektir.

Sonuç olarak Random Forest modelinin eğitiminde kullanılacak `X` özellik matrisi ve `y` hedef vektörü oluşturulacaktır.

Veri kümesinin eğitim ve doğrulama aşamalarında, preprocessing sırasında belirlenen motor bazlı veri ayrımları korunacaktır.

In [15]:
# RF ve XGBoost için ortak özellik çıkarma fonksiyonu
from features.ml_features import extract_window_features

In [16]:
test_features = extract_window_features(
    X_window,
    feature_names
)

print("Özellik sayısı:", len(test_features))

display(test_features.head(12))

Özellik sayısı: 72


alt_mean     10619.550781
alt_std        334.872070
alt_min      10005.000000
alt_max      11309.000000
Mach_mean        0.477126
Mach_std         0.020051
Mach_min         0.447741
Mach_max         0.511749
TRA_mean        77.315735
TRA_std          0.211382
TRA_min         76.903748
TRA_max         77.782646
dtype: float32

### 4. Özellik Matrisi (X) ve Hedef Vektörünün (y) Oluşturulması

Random Forest Regressor modelinin eğitilebilmesi için her pencerenin sabit uzunlukta bir özellik vektörüyle temsil edilmesi gerekmektedir.

Önceki aşamada tanımlanan `extract_window_features()` fonksiyonu kullanılarak tüm pencerelerden istatistiksel özellikler çıkarılacaktır.

Her pencere için hesaplanan 72 özellik, `X` matrisine; pencereye karşılık gelen gerçek RUL değeri ise `y` vektörüne eklenecektir.

Veriler, bellekte gereksiz yer kaplamaması amacıyla pencereler hâlinde okunacak ve yalnızca hesaplanan özellikler saklanacaktır.

Bu aşamada herhangi bir model eğitimi gerçekleştirilmeyecektir.

In [18]:
import importlib
import features.ml_features as ml_features

# Güncellenen Python dosyasını yeniden yükle
importlib.reload(ml_features)

from features.ml_features import (
    extract_window_features,
    build_feature_matrix
)

print("Ortak özellik çıkarma fonksiyonları başarıyla yüklendi.")

Ortak özellik çıkarma fonksiyonları başarıyla yüklendi.


In [19]:
# Tüm pencereler için özellik matrisi ve RUL vektörü
X, y = build_feature_matrix(
    windows,
    feature_names
)

print("X boyutu:", X.shape)
print("y boyutu:", y.shape)

İşlenen pencere: 500/5270
İşlenen pencere: 1000/5270
İşlenen pencere: 1500/5270
İşlenen pencere: 2000/5270
İşlenen pencere: 2500/5270
İşlenen pencere: 3000/5270
İşlenen pencere: 3500/5270
İşlenen pencere: 4000/5270
İşlenen pencere: 4500/5270
İşlenen pencere: 5000/5270
X boyutu: (5270, 72)
y boyutu: (5270,)


### 5. Özellik Matrisi ve Hedef Değerlerin Doğrulanması

Bu aşamada Random Forest modelinin eğitiminde kullanılacak özellik matrisi (`X`) ve hedef vektörü (`y`) kontrol edilmektedir.

Kontroller kapsamında:

- Özellik ve hedef örnek sayılarının eşitliği
- Eksik (`NaN`) ve sonsuz (`Inf`) değerlerin varlığı
- RUL değerlerinin minimum ve maksimum sınırları
- Özellik matrisinin boyutları

incelenmektedir.

Bu kontroller, model eğitiminden önce veri bütünlüğünü doğrulamak amacıyla gerçekleştirilmektedir.

In [20]:
# X ve y boyutlarını kontrol et
print("Örnek sayısı:", len(X))
print("Özellik sayısı:", X.shape[1])
print("Hedef sayısı:", len(y))

# Eksik değer kontrolü
print("\nX eksik değer:", X.isna().sum().sum())
print("y eksik değer:", np.isnan(y).sum())

# Sonsuz değer kontrolü
print("\nX sonsuz değer:", np.isinf(X.to_numpy()).sum())
print("y sonsuz değer:", np.isinf(y).sum())

# RUL değer aralığı
print("\nMinimum RUL:", y.min())
print("Maksimum RUL:", y.max())

# Boyut tutarlılığı
assert len(X) == len(y), "X ve y örnek sayıları eşleşmiyor!"

display(X.head())

Örnek sayısı: 5270
Özellik sayısı: 72
Hedef sayısı: 5270

X eksik değer: 0
y eksik değer: 0

X sonsuz değer: 0
y sonsuz değer: 0

Minimum RUL: 0.0
Maksimum RUL: 88.0


,alt_mean,alt_std,alt_min,alt_max,Mach_mean,Mach_std,Mach_min,Mach_max,TRA_mean,TRA_std,...,Nf_min,Nf_max,Nc_mean,Nc_std,Nc_min,Nc_max,Wf_mean,Wf_std,Wf_min,Wf_max
0,10619.550781,334.872070,10005.0,11309.0,0.477126,0.020051,0.447741,0.511749,77.315735,0.211382,...,2160.716797,2171.427490,8591.742188,3.135091,8585.840820,8596.522461,3.824599,0.013563,3.795418,3.866201
1,17879.730469,433.432098,17149.0,18582.0,0.569451,0.004940,0.560763,0.576135,79.821625,0.158643,...,2144.969727,2150.848633,8448.246094,9.256688,8432.471680,8463.691406,3.004651,0.045695,2.929367,3.082332
2,22709.289062,245.861282,22245.0,23118.0,0.624680,0.003474,0.619164,0.631260,80.992004,0.263722,...,2127.641602,2131.511230,8344.759766,3.314307,8340.642578,8351.888672,2.525351,0.016617,2.500412,2.560521
3,26650.550781,334.027802,26089.0,27175.0,0.648231,0.002011,0.644616,0.653184,81.913467,0.000000,...,2102.947754,2113.099121,8243.814453,11.919615,8225.000000,8262.144531,2.151048,0.036748,2.093489,2.209642
4,30259.488281,201.264084,29900.0,30614.0,0.647718,0.000961,0.646506,0.650601,82.089249,0.000000,...,2074.653564,2080.260986,8125.398438,6.184229,8115.707520,8137.079590,1.803161,0.017189,1.774849,1.835154


### 6. Motor Bazlı Eğitim ve Doğrulama Ayrımı


Aynı motora ait pencereler benzer çalışma koşullarını ve bozulma örüntülerini içerebileceğinden, rastgele pencere bazlı veri ayrımı veri sızıntısına neden olabilir.

Bu nedenle modelin eğitim ve doğrulama aşamalarında preprocessing sürecinde oluşturulan motor bazlı cross-validation ayrımları kullanılacaktır.

Böylece modelin eğitim sırasında görmediği motorlar üzerindeki RUL tahmin performansı değerlendirilebilecektir. Ayrı test kümesi ise nihai değerlendirme için korunacaktır.

In [21]:
import json

# Preprocessing tarafından oluşturulan fold dosyası
fold_path = PROJECT_ROOT / "processed" / "split_folds.json"

with open(fold_path, "r") as f:
    folds = json.load(f)

print("Fold veri tipi:", type(folds))
print("\nFold yapısı:")

if isinstance(folds, dict):
    print("Anahtarlar:", list(folds.keys()))
elif isinstance(folds, list):
    print("Fold sayısı:", len(folds))

print("\nİlk 2000 karakter:")
print(json.dumps(folds, indent=2)[:2000])

Fold veri tipi: <class 'dict'>

Fold yapısı:
Anahtarlar: ['strategy', 'dev_units', 'official_test_units', 'folds']

İlk 2000 karakter:
{
  "strategy": "leave_one_engine_out",
  "dev_units": [
    2,
    5,
    10,
    16,
    18,
    20
  ],
  "official_test_units": [
    11,
    14,
    15
  ],
  "folds": [
    {
      "fold": 1,
      "train_units": [
        5,
        10,
        16,
        18,
        20
      ],
      "validation_units": [
        2
      ]
    },
    {
      "fold": 2,
      "train_units": [
        2,
        10,
        16,
        18,
        20
      ],
      "validation_units": [
        5
      ]
    },
    {
      "fold": 3,
      "train_units": [
        2,
        5,
        16,
        18,
        20
      ],
      "validation_units": [
        10
      ]
    },
    {
      "fold": 4,
      "train_units": [
        2,
        5,
        10,
        18,
        20
      ],
      "validation_units": [
        16
      ]
    },
    {
      "fold": 5,
   

In [ ]:
# Motor bazlı cross-validation ayrımlarını inceleyelim

print("Cross-validation stratejisi:", folds["strategy"])
print("Fold sayısı:", len(folds["folds"]))

for fold in folds["folds"]:
    print(f"\nFold {fold['fold']}")
    print("Eğitim motorları:", fold["train_units"])
    print("Doğrulama motorları:", fold["validation_units"])

Fold anahtarları: ['fold', 'train_units', 'validation_units']

İlk fold'un içeriği:
{
  "fold": 1,
  "train_units": [
    5,
    10,
    16,
    18,
    20
  ],
  "validation_units": [
    2
  ]
}


In [25]:
print("İlk pencere metadata:")
print(windows[0])

print("\nResmi test motorları:")
print(folds["official_test_units"])

İlk pencere metadata:
WindowRecord(window_id=0, unit=2, cycle=1, rul=74.0, start_row=0, end_row=256)

Resmi test motorları:
[11, 14, 15]


In [28]:
# Her pencerenin ait olduğu motor
window_units = np.array([
    window.unit for window in windows
])

# X satırları ile motor kimlikleri aynı sırada olmalı
assert len(window_units) == len(X) == len(y)

# Motor başına pencere sayıları
unit_counts = pd.Series(
    window_units,
    name="Motor"
).value_counts().sort_index()

print("Verideki motorlar:", sorted(np.unique(window_units)))
print("\nMotor başına pencere sayıları:")

display(unit_counts.to_frame("Pencere sayısı"))

# Geliştirme ve test motorları
dev_units = set(folds["dev_units"])
test_units = set(folds["official_test_units"])

# Pencere verilerindeki motorlar
available_units = set(window_units)

print("Geliştirme motorları:", sorted(dev_units))
print("Resmi test motorları:", sorted(test_units))

print(
    "\nVeride bulunan test motorları:",
    sorted(available_units & test_units)
)

print(
    "Veride bulunan geliştirme motorları:",
    sorted(available_units & dev_units)
)

# Eğitim için ayrılmış pencerelerde test motoru olmamalı
assert not (available_units & test_units), (
    "Dikkat! Resmi test motorları geliştirme pencerelerinde bulunuyor."
)

assert available_units == dev_units, (
    "Pencere motorları ile geliştirme motorları eşleşmiyor."
)

Verideki motorlar: [np.int64(2), np.int64(5), np.int64(10), np.int64(16), np.int64(18), np.int64(20)]

Motor başına pencere sayıları:


,Pencere sayısı
Motor,
2,854
5,1034
10,955
16,768
18,890
20,769


Geliştirme motorları: [2, 5, 10, 16, 18, 20]
Resmi test motorları: [11, 14, 15]

Veride bulunan test motorları: []
Veride bulunan geliştirme motorları: [2, 5, 10, 16, 18, 20]


### 7. İlk Fold İçin Eğitim ve Doğrulama Verilerinin Hazırlanması

Leave-One-Engine-Out Cross-Validation stratejisi kapsamında ilk fold seçilmiştir.

Bu fold'da beş motora ait pencereler eğitim verisi, bir motora ait pencereler ise doğrulama verisi olarak kullanılmaktadır.

Motor kimlikleri üzerinden oluşturulan maskeler yardımıyla özellik matrisi (`X`) ve hedef vektörü (`y`) eğitim ve doğrulama alt kümelerine ayrılmaktadır.

Bu yöntem sayesinde aynı motora ait pencerelerin hem eğitim hem de doğrulama kümelerinde bulunması engellenmektedir.

In [29]:
# İlk fold'u seçelim
fold_1 = folds["folds"][0]

train_units = fold_1["train_units"]
val_units = fold_1["validation_units"]

# Motor kimliklerine göre maskeler oluşturalım
train_mask = np.isin(window_units, train_units)
val_mask = np.isin(window_units, val_units)

# Eğitim verileri
X_train = X.loc[train_mask]
y_train = y[train_mask]

# Doğrulama verileri
X_val = X.loc[val_mask]
y_val = y[val_mask]

print("Eğitim motorları:", train_units)
print("Doğrulama motorları:", val_units)

print("\nX_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_val:", X_val.shape)
print("y_val:", y_val.shape)

# Veri sızıntısı kontrolü
assert not set(train_units) & set(val_units)
assert train_mask.sum() + val_mask.sum() == len(X)

Eğitim motorları: [5, 10, 16, 18, 20]
Doğrulama motorları: [2]

X_train: (4416, 72)
y_train: (4416,)

X_val: (854, 72)
y_val: (854,)


### 8. Random Forest Regressor ile İlk Model Eğitimi

Bu aşamada N-CMAPSS DS02 veri kümesinden elde edilen istatistiksel özellikler kullanılarak Random Forest Regressor modeli eğitilmektedir.

İlk deneyde Leave-One-Engine-Out Cross-Validation stratejisinin birinci fold'u kullanılmaktadır.

- **Eğitim verisi:** 5, 10, 16, 18 ve 20 numaralı motorlar
- **Doğrulama verisi:** 2 numaralı motor
- **Girdi değişkenleri:** Her pencere için hesaplanan 72 istatistiksel özellik
- **Hedef değişken:** Remaining Useful Life (RUL)

Random Forest Regressor, birden fazla karar ağacının tahminlerini birleştirerek sürekli sayısal değerler üretir. Bu nedenle RUL tahmini bir regresyon problemi olarak ele alınmaktadır.

İlk model, başlangıç performansını ölçmek amacıyla bir *baseline* olarak eğitilmektedir. Hiperparametre optimizasyonu bu aşamada gerçekleştirilmeyecektir.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,  # 100 karar ağacı
    max_depth=None, # karar ağacının maksimum derinliği
    min_samples_leaf=1, # en az kaç pencere olmalı
    random_state=42, # rastgelelik için seed 
    n_jobs=-1 # paralel işleme
)

print("Random Forest modeli oluşturuldu.")


Random Forest modeli oluşturuldu.


In [32]:
# Random Forest modelini eğitelim
rf_model.fit(X_train, y_train)

print("Random Forest eğitimi tamamlandı.")

Random Forest eğitimi tamamlandı.


In [33]:
# Modelin görmediği motor üzerinde tahmin yapalım
y_pred = rf_model.predict(X_val)

# Gerçek ve tahmin edilen RUL değerlerini karşılaştıralım
results = pd.DataFrame({
    "Gerçek RUL": y_val,
    "Tahmin RUL": y_pred
})

display(results.head(10))

,Gerçek RUL,Tahmin RUL
0,74.0,51.62
1,74.0,53.15
2,74.0,48.46
3,74.0,50.50
4,74.0,47.30
5,74.0,55.63
6,74.0,53.44
7,74.0,52.15
8,74.0,43.33
9,74.0,46.41


In [34]:
rf_model.fit(X_train, y_train)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m